# CrowdStrike Falcon IOC & IOA Migration

This notebook exports **custom IOCs** and **Custom IOA rule groups** from one Falcon CID and imports them into another.

## Prerequisites

1. A `cids.toml` file in this directory with your CID credentials (copy `cids.toml.example` to get started)
2. The `burd` package installed (`uv pip install -e .`)

## How it works

- **Export** reads all IOCs and IOA rule groups from the *source* CID and saves them to JSON files.
- **Import** reads the JSON files and creates them in the *target* CID, skipping any that already exist.
- All imported IOA rule groups and rules are created **disabled** so you can review them before enabling.

## 1. Select CIDs

Load all CID credentials from `cids.toml`, then pick which CID to **export from** (source) and which to **import into** (target).

In [ ]:
import json
import burd

cids = burd.load_cids()

source = burd.select_cid(cids, prompt="Select the SOURCE CID (export from)")
target = burd.select_cid(cids, prompt="Select the TARGET CID (import into)")

## 2. Export

Fetch IOCs and IOA rule groups from the **source** CID and save them to JSON files.
You can inspect or edit the JSON files before importing.

In [ ]:
iocs = burd.export_iocs(source)

In [ ]:
ioa_groups = burd.export_custom_ioas(source)

In [ ]:
with open("iocs.json", "w") as f:
    json.dump(iocs, f, indent=2)
print(f"Wrote iocs.json ({len(iocs)} indicators)")

with open("custom_ioas.json", "w") as f:
    json.dump(ioa_groups, f, indent=2)
print(f"Wrote custom_ioas.json ({len(ioa_groups)} rule groups)")

## 3. Import

Load the JSON files and import into the **target** CID.

**Safety features:**
- Duplicates are detected and skipped automatically (IOCs by type+value, IOA groups by name)
- All IOA rule groups and rules are created **disabled** — enable them manually after review
- Run the **dry run** cells first to see what would be created without making any changes

In [ ]:
with open("iocs.json") as f:
    iocs = json.load(f)
print(f"Loaded {len(iocs)} IOCs from iocs.json")

with open("custom_ioas.json") as f:
    ioa_groups = json.load(f)
print(f"Loaded {len(ioa_groups)} rule groups from custom_ioas.json")

In [ ]:
# Dry run — see what would be created without making changes
burd.import_iocs(target, iocs, dry_run=True)

In [ ]:
# Live import — creates IOCs in the target CID
burd.import_iocs(target, iocs)

In [ ]:
# Dry run — see what would be created without making changes
burd.import_custom_ioas(target, ioa_groups, dry_run=True)

In [ ]:
# Live import — creates IOA rule groups and rules in the target CID
burd.import_custom_ioas(target, ioa_groups)

## 4. Duplicate Host Cleanup

Find and remove duplicate host/sensor records within a single CID.
Duplicates are identified by matching `(hostname, mac_address)` — the host with the most recent `last_seen` timestamp is kept.

In [ ]:
dedup_cid = burd.select_cid(cids, prompt="Select the CID to deduplicate")

In [ ]:
duplicates, kept = burd.find_duplicate_hosts(dedup_cid)

In [ ]:
# Dry run — see what would be hidden without making changes
burd.hide_duplicate_hosts(dedup_cid, duplicates, dry_run=True)

In [ ]:
# Live hide — removes duplicate hosts from the CID
burd.hide_duplicate_hosts(dedup_cid, duplicates)